In [ ]:
Mandi Data api Key = 579b464db66ec23bdd000001e3a48235641a458b502501e95c36f78e

Mandi Data file = "D:\CUDA_Experiments\Git_HUB\AgriCast360\Script\Mandi_Data.csv"

In [ ]:
Weatherbit API

Example API URL: https://api.weatherbit.io/v2.0/history/subhourly?lat=35.78&lon=78.64&start_date=2025-11-11&end_date=2025-11-12&key=API_KEY

Master API Key = de8e274a65ec45929d501a63466da6cf	

(Plan	Details	Support Level
Business Trial (Expires 2025-12-03)
1,500 req/day
1,500 historical req/day
25 years historical
Current weather + alerts + lightning
Daily forecasts
Hourly forecasts
60 minute forecasts
+ Energy / Air Quality / Agweather / Climate Normals API
Non-Commercial use only)

In [2]:
import requests
import pandas as pd
from datetime import timedelta
import time
import sys
import json
import os
from tqdm import tqdm

# --- CONFIGURATION ---

# 1. ADD YOUR API KEYS HERE
# It will rotate through this list if one key hits its rate limit.
API_KEYS = [
    "de8e274a65ec45929d501a63466da6cf", # Business Trial 1 (Expires 2025-12-03)
    "7335451fcc0646bb86611027b68292e0", # Business Trial 2 (Expires 2025-12-07)
    "9e56a4f3fab748418bc299713138f15c", # Business Trial 3 (Expires 2025-12-07)
    "fa8e8af08314f64ab0d2f7492b46e9f", # Business Trial 4 (Expires 2025-12-07)
    "680cfd56aefc45d6bfa1d40fb65a3028", # Business Trial 5 (Expires 2025-12-08)
    "a1d90659369044278f0d29c1497dfa54",  # NEW KEY ml.agricast@gmail.com
    "f9753308410b414a96a4a7081e68444e" # KD Key
]
# Global index to track which key we are currently using
CURRENT_KEY_INDEX = 1

# 2. ADD YOUR MARKETS HERE
# The script will loop through these and save a CSV for each.
MARKETS = {
    "Bardoli": {"lat": 21.1439076246341, "lon": 73.249972681491997},
    "Bardoli_Katod": {"lat": 21.12, "lon": 73.12},
    "Bardoli_Madhi": {"lat": 21.15, "lon": 73.25},
    "Kosamba": {"lat": 21.464459923454399, "lon": 72.952089010333296},
    "Kosamba_Vankal": {"lat": 21.43, "lon": 73.23},
    "Kosamba_Zangvav": {"lat": 21.48, "lon": 72.95},
    "Mahuva": {"lat": 21.097129557468101, "lon": 71.760557527893894},
    "Mahuva_Anaval": {"lat": 20.84, "lon": 73.26},
    "Mandvi": {"lat": 21.259469954916099, "lon": 73.306064794408599},
    "Nizar": {"lat": 21.47, "lon": 74.19},
    "Nizar_Kukarmuda": {"lat": 21.51, "lon": 74.13},
    "Nizar_Pumkitalov": {"lat": 21.47, "lon": 74.10},
    "Songadh": {"lat": 21.175868577082898, "lon": 73.564033008192197},
    "Songadh_Badarpada": {"lat": 21.16, "lon": 73.56},
    "Songadh_Umrada": {"lat": 21.16, "lon": 73.56},
    "Surat": {"lat": 21.193417012914299, "lon": 72.852982768754401},
    "Uchhal": {"lat": 21.17, "lon": 73.74},
    "Valod_Buhari": {"lat": 20.97, "lon": 73.31},
    "Vyara_Paati": {"lat": 21.11, "lon": 73.38},
    "Vyra": {"lat": 21.112412216859902, "lon": 73.388557572057493},
    "Amreli": {"lat": 21.559338614206901, "lon": 71.227430257825006},
    "Babra": {"lat": 21.848355060711398, "lon": 71.311297950245503},
    "Bagasara": {"lat": 21.496945117777699, "lon": 70.959329215009802},
    "Dhari": {"lat": 21.331365216974699, "lon": 71.023828198856805},
    "Rajula": {"lat": 21.0339169150099, "lon": 71.443913765569206},
    "Savarkundla": {"lat": 21.332621199094, "lon": 71.313830460040407},
    "Ahmedabad_Chimanbhai_Patal_Market_Vasana": {"lat": 22.996976855033001, "lon": 72.536392460074595},
    "Bavla": {"lat": 22.830965521422801, "lon": 72.3579378735662},
    "Dhandhuka": {"lat": 22.377117706820101, "lon": 71.978523283421794},
    "Dholka": {"lat": 22.7212505077608, "lon": 72.448905310358995},
    "Mandal": {"lat": 23.282162015159599, "lon": 71.915345532018506},
    "Sanad": {"lat": 22.997609695821499, "lon": 72.381634273951903},
    "Viramgam": {"lat": 23.125945592319901, "lon": 72.045712191155303}
}

# 3. SETTINGS
BASE_URL = "https://api.weatherbit.io/v2.0/history/daily"
START_DATE = '2024-01-01'
END_DATE = '2024-12-31'

# Columns to remove from the final CSV
COLUMNS_TO_DROP = ['snow_rate', 'weather.icon', 'revision_version', 'timestamp_utc']

# File to track progress
PROGRESS_FILE = 'scrape_progress.json'

# --- HELPER FUNCTIONS ---

def get_api_key():
    """
    Gets the current API key and advances the index for rotation.
    Returns None if all keys are exhausted.
    """
    global CURRENT_KEY_INDEX
    if CURRENT_KEY_INDEX >= len(API_KEYS):
        return None  # All keys have been tried and failed
    
    key = API_KEYS[CURRENT_KEY_INDEX]
    return key

def rotate_to_next_key():
    """
    Moves to the next key in the list.
    Returns True if a new key is available, False if all keys are exhausted.
    """
    global CURRENT_KEY_INDEX
    CURRENT_KEY_INDEX += 1
    if CURRENT_KEY_INDEX >= len(API_KEYS):
        return False  # No more keys to try
    return True

def load_progress():
    """Loads the progress file if it exists."""
    if os.path.exists(PROGRESS_FILE):
        print(f"Loading progress from {PROGRESS_FILE}...")
        try:
            with open(PROGRESS_FILE, 'r') as f:
                return json.load(f)
        except json.JSONDecodeError:
            print("Warning: Progress file is corrupt. Starting from scratch.")
            return {}
    print("No progress file found, starting from scratch.")
    return {}

def save_progress(progress):
    """Saves the current progress to the checkpoint file."""
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, indent=4)

def parse_date(date_str):
    """Converts YYYY-MM-DD string to datetime object."""
    return pd.to_datetime(date_str)

def format_date(date_obj):
    """Converts datetime object to YYYY-MM-DD string."""
    return date_obj.strftime('%Y-%m-%d')

def save_data_to_csv(new_data_frames, output_filename):
    """
    Appends new data to the CSV file, handling duplicates and headers.
    """
    if not new_data_frames:
        print("\nNo new data to save in this batch.")
        return 'SUCCESS'

    print(f"\nSaving {len(new_data_frames)} new records to CSV...")
    
    try:
        new_df = pd.concat(new_data_frames, ignore_index=True)
    except Exception as e:
        print(f"Error concatenating new dataframes: {e}")
        return 'SUCCESS' # Nothing to save

    combined_df = new_df

    # Load existing data (if any) and combine
    if os.path.exists(output_filename):
        try:
            existing_df = pd.read_csv(output_filename)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            print(f"Loaded {len(existing_df)} existing records. Total records before de-dupe: {len(combined_df)}")
        except pd.errors.EmptyDataError:
            print(f"Existing file {output_filename} is empty. Using new data.")
        except Exception as e:
            print(f"Error reading {output_filename}: {e}. Overwriting with new data.")
    else:
        print("Creating new file...")

    # De-duplicate, keeping the last (newest) record for any given day
    combined_df.drop_duplicates(subset=['market_name', 'datetime'], keep='last', inplace=True)
    
    # Drop unwanted columns
    existing_cols_to_drop = [col for col in COLUMNS_TO_DROP if col in combined_df.columns]
    if existing_cols_to_drop:
        combined_df.drop(columns=existing_cols_to_drop, inplace=True)
    
    # Save to CSV
    try:
        combined_df.to_csv(output_filename, index=False)
        print(f"Successfully saved {len(combined_df)} total unique records to {output_filename}")
        return 'SUCCESS'
    except PermissionError:
        print(f"\nCRITICAL: Could not save {output_filename}. Is the file open in Excel?")
        print("Please close the file and re-run the script.")
        return 'STOP_FROM_SAVE'
    except Exception as e:
        print(f"\nCRITICAL: Failed to save CSV: {e}")
        return 'STOP_FROM_SAVE'


def fetch_weather_data(market_name, lat, lon, start_date, end_date, progress):
    """
    Fetches weather data for a specific market and date range.
    Handles API key rotation and rate limiting.
    """
    
    # Use pandas to create the list of days to query
    all_days = pd.date_range(start_date, end_date, freq='D')
    total_days = len(all_days)
    
    # Get filename for this market's CSV
    output_filename = f"{market_name.replace(' ', '_')}_weather_{start_date}_to_{end_date}.csv"
    
    # This list will ONLY hold NEW data fetched in this run
    new_data_fetched_this_run = [] 
    if os.path.exists(output_filename):
        print(f"File {output_filename} already exists. New data will be appended.")
    else:
        print(f"Creating new file: {output_filename}")

    # --- Progress Loading Logic ---
    market_progress_data = progress.get(market_name, {})
    last_fetched_day_str = None
    market_status = None

    if isinstance(market_progress_data, dict):
        last_fetched_day_str = market_progress_data.get('last_fetched_day')
        market_status = market_progress_data.get('status')
    elif isinstance(market_progress_data, str):
        if market_progress_data == 'completed':
            market_status = 'completed'
            last_fetched_day_str = None
        else:
            last_fetched_day_str = market_progress_data
            print("Detected old progress format. Will update to new format.")
    
    if market_status == 'completed':
        print(f"Market {market_name} is already marked as 'completed'. Skipping.")
        return 'CONTINUE'

    # Determine where to start fetching from
    if last_fetched_day_str:
        start_fetching_from = parse_date(last_fetched_day_str) + timedelta(days=1)
        days_to_query = [day for day in all_days if day >= start_fetching_from]
        if start_fetching_from > parse_date(end_date):
            print("All days already fetched. Marking as complete.")
            days_to_query = []
        else:
            print(f"Resuming from {format_date(start_fetching_from)}...")
    else:
        days_to_query = all_days
        print(f"Starting from {start_date}...")

    # FIX 1: Handle ValueError for DatetimeIndex
    if len(days_to_query) == 0:
        print("No new days to query. Marking as complete.")
        progress[market_name] = {'status': 'completed', 'last_fetched_day': end_date}
        save_progress(progress)
        return 'CONTINUE'

    print(f"Total days to query for this market: {len(days_to_query)}.")

    # --- MAIN FETCHING LOOP (RESTRUCTURED) ---
    with tqdm(total=len(days_to_query), desc=f"Fetching for {market_name}") as pbar:
        for day in days_to_query:
            
            # This inner loop will retry the *same day* until it
            # succeeds or all keys are exhausted.
            day_fetched_successfully = False
            
            # FIX 2: Added 'while' loop for retries
            while not day_fetched_successfully:
                
                api_key = get_api_key()
                if api_key is None:
                    # All keys are exhausted for today.
                    print("\nCRITICAL: All API keys are exhausted. Stopping script.")
                    print("Saving partial data before stopping...")
                    save_status = save_data_to_csv(new_data_fetched_this_run, output_filename)
                    if save_status == 'STOP_FROM_SAVE':
                        print("Could not save partial data. Data will be re-fetched on next run.")
                    else:
                        print("Partial data saved. Progress will be updated.")
                        save_progress(progress) 
                    
                    print("Run the script again tomorrow (or add new keys) to resume.")
                    return 'STOP' # Stop the entire batch process
                
                pbar.set_description(f"Fetching for {market_name} [{day.strftime('%Y-%m-%d')}]", refresh=True)
                pbar.set_postfix_str(f"Key: ...{api_key[-4:]}")

                day_start_str = format_date(day)
                day_end_str = format_date(day + timedelta(days=1))

                params = {
                    'key': api_key,
                    'lat': lat,
                    'lon': lon,
                    'start_date': day_start_str,
                    'end_date': day_end_str
                }

                try:
                    response = requests.get(BASE_URL, params=params, timeout=10)

                    # --- Rate Limit / Auth Error ---
                    # This now correctly handles 403 (Forbidden),
                    # which your new provisioning key might return.
                    if response.status_code == 429 or response.status_code == 403:
                        print(f"\nWarning: Key ...{api_key[-4:]} failed (Rate Limit/Auth). Trying next key.")
                        if not rotate_to_next_key():
                            # This was the last key. get_api_key() will return None
                            # on the next 'while' loop iteration, triggering the STOP.
                            pass
                        
                        time.sleep(1)
                        # 'continue' will retry the 'while' loop for the SAME DAY
                        continue 

                    # --- Other HTTP Error ---
                    if response.status_code != 200:
                        print(f"\nError: Received status code {response.status_code} for {day_start_str}.")
                        print(f"Response: {response.text}")
                        time.sleep(1)
                        # Break the 'while' loop and skip to the next day.
                        # This day will be retried on the next script run.
                        break 

                    # --- Success ---
                    data = response.json()
                    daily_data = data.get('data', [])
                    
                    if daily_data:
                        df = pd.json_normalize(daily_data)
                        df['market_name'] = market_name
                        df['query_lat'] = lat
                        df['query_lon'] = lon
                        new_data_fetched_this_run.append(df)
                    
                    progress[market_name] = {'last_fetched_day': day_start_str}
                    save_progress(progress)
                    
                    day_fetched_successfully = True # This breaks the 'while' loop
                    pbar.update(1) # Advance progress bar by one day
                    time.sleep(0.1) # Be nice to the API

                except requests.exceptions.RequestException as e:
                    print(f"\nNetwork Error: {e}. Pausing for 10 seconds...")
                    time.sleep(10)
                    # 'continue' will retry the 'while' loop for the SAME DAY
                    continue
            
            # --- End of while loop ---
            # The 'for' loop will now proceed to the next day.

    # --- Market Finished ---
    save_status = save_data_to_csv(new_data_fetched_this_run, output_filename)
    if save_status == 'STOP_FROM_SAVE':
        print(f"Could not save data for {market_name}. Market will NOT be marked as complete.")
        return 'STOP' 

    progress[market_name] = {'status': 'completed', 'last_fetched_day': end_date}
    save_progress(progress)
    print(f"--- Finished processing {market_name} ---")
    return 'CONTINUE'

def main():
    """
    Main function to run the batch scraper.
    """
    print("--- Starting Weatherbit Batch Scraper ---")
    
    # Check if argparse is being used in an interactive env (like Jupyter)
    # This ignores Jupyter's internal args like --f
    if 'ipykernel_launcher.py' in sys.argv[0]:
        sys.argv = [sys.argv[0]]  # Keep only the script name

    # (argparse setup is removed as we are hardcoding markets)

    print(f"Found {len(MARKETS)} markets to process.")

    # Load progress from checkpoint file
    progress = load_progress()

    # Loop over each market defined in the config
    for market_name, coords in MARKETS.items():
        
        # --- Backwards-Compatibility Check ---
        market_progress_data = progress.get(market_name, {})
        market_status = None
        if isinstance(market_progress_data, dict):
            market_status = market_progress_data.get('status')
        elif isinstance(market_progress_data, str):
            if market_progress_data == 'completed':
                market_status = 'completed'
            else:
                market_status = None 
                print(f"Detected old progress format for {market_name}. Will update.")

        if market_status == 'completed':
            print(f"Market {market_name} is already 'completed'. Skipping.")
            continue
            
        print(f"\n--- Processing Market: {market_name} ---")
        
        status = fetch_weather_data(
            market_name=market_name,
            lat=coords['lat'],
            lon=coords['lon'],
            start_date=START_DATE,
            end_date=END_DATE,
            progress=progress # Pass the main progress object
        )
        
        if status == 'STOP':
            print("\nBatch process stopped.")
            print("Run the script again later to resume from the last checkpoint.")
            break # Stop processing other markets
    
    print("--- Batch processing finished. ---")

if __name__ == "__main__":
    main()

--- Starting Weatherbit Batch Scraper ---
Found 33 markets to process.
Loading progress from scrape_progress.json...
Market Bardoli is already 'completed'. Skipping.
Market Bardoli_Katod is already 'completed'. Skipping.
Market Bardoli_Madhi is already 'completed'. Skipping.
Market Kosamba is already 'completed'. Skipping.
Market Kosamba_Vankal is already 'completed'. Skipping.
Market Kosamba_Zangvav is already 'completed'. Skipping.
Market Mahuva is already 'completed'. Skipping.
Market Mahuva_Anaval is already 'completed'. Skipping.
Market Mandvi is already 'completed'. Skipping.
Market Nizar is already 'completed'. Skipping.
Market Nizar_Kukarmuda is already 'completed'. Skipping.
Market Nizar_Pumkitalov is already 'completed'. Skipping.
Market Songadh is already 'completed'. Skipping.
Market Songadh_Badarpada is already 'completed'. Skipping.
Market Songadh_Umrada is already 'completed'. Skipping.
Market Surat is already 'completed'. Skipping.
Market Uchhal is already 'completed'. 

Fetching for Dholka [2024-03-31]:   0%|          | 0/276 [00:00<?, ?it/s, Key: ...92e0]

Fetching for Dholka [2024-03-31]:   0%|          | 0/276 [00:01<?, ?it/s, Key: ...f15c]

Fetching for Dholka [2024-03-31]:   0%|          | 0/276 [00:03<?, ?it/s, Key: ...6e9f]

Fetching for Dholka [2024-03-31]:   0%|          | 0/276 [00:04<?, ?it/s, Key: ...3028]

...


Saving 366 new records to CSV...
Creating new file...
Successfully saved 366 total unique records to Sanad_weather_2024-01-01_to_2024-12-31.csv
--- Finished processing Sanad ---

--- Processing Market: Viramgam ---
Creating new file: Viramgam_weather_2024-01-01_to_2024-12-31.csv
Starting from 2024-01-01...
Total days to query for this market: 366.


Fetching for Viramgam [2024-05-24]:  39%|███▉      | 144/366 [01:35<02:24,  1.53it/s, Key: ...fa54]


Network Error: HTTPSConnectionPool(host='api.weatherbit.io', port=443): Max retries exceeded with url: /v2.0/history/daily?key=a1d90659369044278f0d29c1497dfa54&lat=23.1259455923199&lon=72.0457121911553&start_date=2024-05-24&end_date=2024-05-25 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000262A4839510>, 'Connection to api.weatherbit.io timed out. (connect timeout=10)')). Pausing for 10 seconds...


Fetching for Viramgam [2024-12-31]: 100%|██████████| 366/366 [04:21<00:00,  1.40it/s, Key: ...fa54]


Saving 366 new records to CSV...
Creating new file...
Successfully saved 366 total unique records to Viramgam_weather_2024-01-01_to_2024-12-31.csv
--- Finished processing Viramgam ---
--- Batch processing finished. ---


Mandi Data Cleaner

In [2]:
import os
import pandas as pd
import glob

# Configuration
SOURCE_FOLDER = r"D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Mandi Data"

# Define the mapping of Old Name -> New Name
COMMODITY_MAPPING = {
    "Cucumbar(Kheera)": "Cucumbar",
    "Mango (Raw-Ripe)": "Mango (Raw)",
    "French Beans (Frasbean)": "French Beans",
    "Pointed gourd (Parval)": "Parval",
    "Elephant Yam (Suran)": "Suran",
    "Surat Beans (Papadi)": "Papadi",
    "Cummin Seed(Jeera)": "Jeera",
    "Isabgul (Psyllium)": "Isabgul",
    "Sesamum(Sesame,Gingelly,Til)": "Til",
    "Paddy(Dhan)(Common)": "Paddy(Dhan)",
    "Paddy(Dhan)(Common)(Whole)": "Paddy(Dhan)",
    "Suva (Dill Seed)": "Suva",
    "Amla(Nelli Kai)": "Amla",
    "Arhar (Tur/Red Gram)(Whole)": "Arhar",
    "Arhar (Tur/Red Gram)": "Arhar",
    "Bajra(Pearl Millet/Cumbu)": "Bajra",
    "Bengal Gram(Gram)(Whole)": "Bengal Gram",
    "Black Gram (Urd Beans)(Whole)": "Urd Beans",
    "Black Gram (Urd Beans)": "Urd Beans",
    "Coriander(Leaves)": "Coriander (Leaves)",
    "Corriander seed": "Coriander (seed)",
    "Guar Seed(Cluster Beans Seed)": "Guar Seed",
    "Methi(Leaves)": "Methi (Leaves)",
    "Methi Seeds": "Methi (Seeds)",
    "Mint(Pudina)": "Pudina",
    "Rat Tail Radish (Mogari)": "Mogari",
    "Yam (Ratalu)": "Ratalu",
    "Banana - Green": "Banana (G)",
    "Bhindi(Ladies Finger)": "Bhindi",
    "Green Gram (Moong)(Whole)": "Moong",
    "Green Gram (Moong)": "Moong",
    "Jowar(Sorghum)": "Jowar",
    "Kabuli Chana(Chickpeas-White)": "Kabuli Chana",
    "Kulthi(Horse Gram)": "Kulthi",
    "Cowpea (Lobia/Karamani)": "Cowpea",
    "Cowpea(Veg)": "Cowpea",
    "Gram Raw(Chholia)": "Chholia",
    "Kartali (Kantola)": "Kantola",
    "Lentil (Masur)": "Masur",
    "Little gourd (Kundru)": "Kundru",
    "Pegeon Pea (Arhar Fali)": "Arhar Fali",
    "Ridgeguard(Tori)": "Tori"
}

def process_excel_files(directory):
    """
    Iterates through all .xlsx files in the directory and applies string replacements.
    """
    # Verify directory exists
    if not os.path.exists(directory):
        print(f"Error: Directory not found: {directory}")
        return

    # Get list of xlsx files
    files = glob.glob(os.path.join(directory, "*.xlsx"))
    
    if not files:
        print(f"No .xlsx files found in {directory}")
        return

    print(f"Found {len(files)} files. Starting processing...")

    count = 0
    for filepath in files:
        try:
            filename = os.path.basename(filepath)
            print(f"Processing: {filename}...")

            # Read the Excel file
            df = pd.read_excel(filepath)

            # Apply replacements across the entire dataframe
            # This handles the mapping efficiently regardless of column name
            df = df.replace(COMMODITY_MAPPING)

            # Save the file back (Overwriting the original)
            # index=False ensures we don't add an extra index column
            df.to_excel(filepath, index=False)
            
            count += 1
            
        except Exception as e:
            print(f"Failed to process {filename}: {str(e)}")

    print(f"\nProcessing complete. Successfully updated {count}/{len(files)} files.")

if __name__ == "__main__":
    # Ensure you have openpyxl installed: pip install pandas openpyxl
    process_excel_files(SOURCE_FOLDER)

Found 4 files. Starting processing...
Processing: Mandi_Ahmedabad.xlsx...
Processing: Mandi_Amreli.xlsx...
Processing: Mandi_Surat.xlsx...
Processing: Total_Comodity.xlsx...

Processing complete. Successfully updated 4/4 files.


Rename Commodity Script

In [3]:
import pandas as pd
import os
import glob

def process_excel_files(directory_path):
    # Check if directory exists
    if not os.path.exists(directory_path):
        print(f"Error: Directory not found at {directory_path}")
        return

    # Find all Excel files (.xlsx and .xls)
    files = glob.glob(os.path.join(directory_path, "*.xlsx")) + glob.glob(os.path.join(directory_path, "*.xls"))
    
    if not files:
        print("No Excel files found in the specified directory.")
        return

    print(f"Found {len(files)} files. Starting processing...")

    for file_path in files:
        try:
            # Read the Excel file
            df = pd.read_excel(file_path)
            
            # Check if required columns exist
            if 'Commodity' not in df.columns or 'Commodity_Code' not in df.columns:
                print(f"Skipping {os.path.basename(file_path)}: Missing required columns.")
                continue

            # Define the condition: Commodity is 'Cowpea' AND Commodity_Code is 89
            # We cast Commodity_Code to numeric to ensure '89' (string) and 89 (int) are treated similarly if needed,
            # but strictly adhering to the request, we compare against 89.
            condition = (df['Commodity'] == 'Cowpea') & (df['Commodity_Code'] == 89)
            
            # Count rows to be changed for logging
            rows_to_change = condition.sum()
            
            if rows_to_change > 0:
                # Apply the update
                df.loc[condition, 'Commodity'] = 'Cowpea (P)'
                
                # Save the file back, overwriting the original
                # index=False prevents pandas from adding a new index column
                df.to_excel(file_path, index=False)
                
                print(f"Updated {rows_to_change} rows in: {os.path.basename(file_path)}")
            else:
                print(f"No matching rows found in: {os.path.basename(file_path)}")

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

if __name__ == "__main__":
    # The path provided in your prompt
    folder_path = r"D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Mandi Data"
    
    process_excel_files(folder_path)

Found 7 files. Starting processing...
No matching rows found in: Mandi_Ahmedabad.xlsx
No matching rows found in: Mandi_Amreli.xlsx
Updated 456 rows in: Mandi_Surat.xlsx
Skipping Total_Comodity.xlsx: Missing required columns.
Error processing ~$Mandi_Ahmedabad.xlsx: [Errno 2] No such file or directory: 'D:\\CUDA_Experiments\\Git_HUB\\Machine-Learning\\AgriCast360\\AgriCast360_V2\\Mandi Data\\~$Mandi_Ahmedabad.xlsx'
Error processing ~$Mandi_Surat.xlsx: [Errno 2] No such file or directory: 'D:\\CUDA_Experiments\\Git_HUB\\Machine-Learning\\AgriCast360\\AgriCast360_V2\\Mandi Data\\~$Mandi_Surat.xlsx'
Error processing ~$Total_Comodity.xlsx: [Errno 13] Permission denied: 'D:\\CUDA_Experiments\\Git_HUB\\Machine-Learning\\AgriCast360\\AgriCast360_V2\\Mandi Data\\~$Total_Comodity.xlsx'


Weather Data Processing

In [6]:
import pandas as pd
import os
import glob

def process_weather_files(directory_path):
    # Check if directory exists
    if not os.path.exists(directory_path):
        print(f"Error: Directory not found at {directory_path}")
        return

    # Find all CSV files
    files = glob.glob(os.path.join(directory_path, "*.csv"))
    
    if not files:
        print("No CSV files found in the specified directory.")
        return

    print(f"Found {len(files)} files. Starting processing...")

    # Define columns to remove
    cols_to_drop = ['max_temp_ts', 'max_wind_spd_ts', 'min_temp_ts', 'ts', 'snow', 'snow_depth', 'revision_status (Text)','t_dhi (Unix time)', 't_dni (Unix time)', 't_ghi (Unix time)', 't_solar_rad (Unix time)']
    
    # Define mapping for renaming columns with units
    # Using specific units from the provided table
    unit_mapping = {
        'clouds': 'clouds (%)',
        'datetime': 'Date ',
        'dewpt': 'dewpt (°C)',
        'dhi': 'dhi (W/m²)',
        'dni': 'dni (W/m²)',
        'ghi': 'ghi (W/m²)',
        'max_dhi': 'max_dhi (W/m²)',
        'max_dni': 'max_dni (W/m²)',
        'max_ghi': 'max_ghi (W/m²)',
        'max_temp': 'max_temp (°C)',
        'max_uv': 'max_uv (Index)',
        'max_wind_dir': 'max_wind_dir (°)',
        'max_wind_spd': 'max_wind_spd (m/s)',
        'min_temp': 'min_temp (°C)',
        'precip': 'precip (mm)',
        'precip_gpm': 'precip_gpm (mm/hr)',
        'pres': 'pres (hPa)',
        'revision_status': 'revision_status (Text)',
        'rh': 'rh (%)',
        'slp': 'slp (hPa)',
        'solar_rad': 'solar_rad (W/m²)',
        't_dhi': 't_dhi (Unix time)',
        't_dni': 't_dni (Unix time)',
        't_ghi': 't_ghi (Unix time)',
        't_solar_rad': 't_solar_rad (Unix time)',
        'temp': 'temp (°C)',
        'wind_dir': 'wind_dir (°)',
        'wind_gust_spd': 'wind_gust_spd (m/s)',
        'wind_spd': 'wind_spd (m/s)',
        'market_name': 'market_name (Text)',
        'query_lat': 'query_lat (°)',
        'query_lon': 'query_lon (°)'
    }
    
    # Define the string to remove from filename
    # Assuming the user meant the text inside the brackets. 
    # If the file literally has brackets like "City(_weather...).csv", change this to "(_weather_2024-01-01_to_2024-12-31)"
    string_to_remove = "_weather_2024-01-01_to_2024-12-31"

    for file_path in files:
        try:
            # 1. Read the CSV file
            df = pd.read_csv(file_path)
            
            # 2. Divide 'clouds' by 100 if it exists
            if 'clouds' in df.columns:
                # df['clouds'] = df['clouds'] / 100
                df['rh (%)'] = df['rh (%)'] / 100 
            
            # 3. Remove specified columns (errors='ignore' prevents crashing if a column is missing)
            df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
            
            # 4. Rename remaining columns with units
            df.rename(columns=unit_mapping, inplace=True)
            
            # 5. Handle File Renaming
            directory, filename = os.path.split(file_path)
            
            # Remove the specific string from the filename
            new_filename = filename.replace(string_to_remove, "")
            
            # Clean up potential double underscores or trailing underscores left behind
            if new_filename.endswith("_.csv"):
                new_filename = new_filename.replace("_.csv", ".csv")
            
            new_file_path = os.path.join(directory, new_filename)
            
            # 6. Save the modified data
            # If the name hasn't changed, this overwrites. If it has, this creates a new file.
            df.to_csv(new_file_path, index=False)
            
            # 7. Cleanup: If we created a new file name, remove the old file
            if new_file_path != file_path:
                os.remove(file_path)
                print(f"Processed & Renamed: {filename} -> {new_filename}")
            else:
                print(f"Processed (No Rename): {filename}")

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

if __name__ == "__main__":
    folder_path = r"D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Weather Data"
    process_weather_files(folder_path)

Found 33 files. Starting processing...
Error processing Ahmedabad_Chimanbhai_Patal_Market_Vasana.csv: [Errno 13] Permission denied: 'D:\\CUDA_Experiments\\Git_HUB\\Machine-Learning\\AgriCast360\\AgriCast360_V2\\Weather Data\\Ahmedabad_Chimanbhai_Patal_Market_Vasana.csv'
Processed (No Rename): Amreli.csv
Processed (No Rename): Babra.csv
Processed (No Rename): Bagasara.csv
Processed (No Rename): Bardoli_Katod.csv
Processed (No Rename): Bardoli_Madhi.csv
Error processing Bardoli_weather_2024-01-01_to_2024-12-31.csv: 'utf-8' codec can't decode byte 0xb0 in position 30: invalid start byte
Processed (No Rename): Bavla.csv
Processed (No Rename): Dhandhuka.csv
Processed (No Rename): Dhari.csv
Processed (No Rename): Dholka.csv
Processed (No Rename): Kosamba.csv
Processed (No Rename): Kosamba_Vankal.csv
Processed (No Rename): Kosamba_Zangvav.csv
Processed (No Rename): Mahuva.csv
Processed (No Rename): Mahuva_Anaval.csv
Processed (No Rename): Mandal.csv
Processed (No Rename): Mandvi.csv
Processed

In [7]:
import pandas as pd
import os
import glob

def fix_rh_values(directory_path):
    # Check if directory exists
    if not os.path.exists(directory_path):
        print(f"Error: Directory not found at {directory_path}")
        return

    # Find all CSV files
    files = glob.glob(os.path.join(directory_path, "*.csv"))
    
    if not files:
        print("No CSV files found in the specified directory.")
        return

    print(f"Found {len(files)} files. Checking for 'rh (%)' column to update...")

    for file_path in files:
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)
            
            # Check if the specific column exists
            if 'rh (%)' in df.columns:
                # Apply the division
                # We check a sample value to ensure we don't divide twice (e.g., if max > 1, it likely needs division)
                # However, to strictly follow instructions, we will simply divide.
                # If you want safety checks (e.g. don't divide if max is already < 1), uncomment the check below.
                
                # if df['rh (%)'].max() > 1.0: 
                df['rh (%)'] = df['rh (%)'] / 100
                
                # Save the file back
                df.to_csv(file_path, index=False)
                print(f"Updated 'rh (%)' in: {os.path.basename(file_path)}")
            else:
                print(f"Skipped (Column 'rh (%)' not found): {os.path.basename(file_path)}")

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

if __name__ == "__main__":
    folder_path = r"D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Weather Data"
    fix_rh_values(folder_path)

Found 33 files. Checking for 'rh (%)' column to update...
Updated 'rh (%)' in: Ahmedabad_Chimanbhai_Patal_Market_Vasana.csv
Updated 'rh (%)' in: Amreli.csv
Updated 'rh (%)' in: Babra.csv
Updated 'rh (%)' in: Bagasara.csv
Updated 'rh (%)' in: Bardoli_Katod.csv
Updated 'rh (%)' in: Bardoli_Madhi.csv
Error processing Bardoli_weather_2024-01-01_to_2024-12-31.csv: 'utf-8' codec can't decode byte 0xb0 in position 30: invalid start byte
Updated 'rh (%)' in: Bavla.csv
Updated 'rh (%)' in: Dhandhuka.csv
Updated 'rh (%)' in: Dhari.csv
Updated 'rh (%)' in: Dholka.csv
Updated 'rh (%)' in: Kosamba.csv
Updated 'rh (%)' in: Kosamba_Vankal.csv
Updated 'rh (%)' in: Kosamba_Zangvav.csv
Updated 'rh (%)' in: Mahuva.csv
Updated 'rh (%)' in: Mahuva_Anaval.csv
Updated 'rh (%)' in: Mandal.csv
Updated 'rh (%)' in: Mandvi.csv
Updated 'rh (%)' in: Nizar.csv
Updated 'rh (%)' in: Nizar_Kukarmuda.csv
Updated 'rh (%)' in: Nizar_Pumkitalov.csv
Updated 'rh (%)' in: Rajula.csv
Updated 'rh (%)' in: Sanad.csv
Updated 'rh

In [5]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

# Input CSV path
csv_path = r"D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Model_Results\16_price_predictions_with_metadata.csv"
output_dir = os.path.dirname(csv_path)
html_path = os.path.join(output_dir, "price_predictions_report.html")

# Read CSV
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} records with {len(df.columns)} columns")

# === Extract Models & Compute Metrics ===
# Identify model names from column suffixes (_Predicted, _Diff, _AbsDiff)
model_names = []
for col in df.columns:
    if col.endswith("_Predicted"):
        model_names.append(col.replace("_Predicted", ""))

# Compute per-model MAE and RMSE
model_metrics = {}
for model in model_names:
    abs_diff_col = f"{model}_AbsDiff"
    if abs_diff_col in df.columns:
        mae = df[abs_diff_col].mean()
        rmse = np.sqrt((df[abs_diff_col] ** 2).mean())
        model_metrics[model] = {"MAE": mae, "RMSE": rmse}

# Sort by MAE
sorted_models = sorted(model_metrics.items(), key=lambda x: x[1]["MAE"])

# === Prepare Data for Filters & Charts ===
unique_commodities = sorted(df['Commodity'].unique().tolist())
unique_markets = sorted(df['Market'].unique().tolist())

# Compute error stats by commodity
commodity_stats = {}
for comm in unique_commodities:
    comm_data = df[df['Commodity'] == comm]
    commodity_stats[comm] = {
        "count": len(comm_data),
        "avg_actual": comm_data['Actual_Price'].mean(),
        "avg_error": comm_data[[f"{m}_AbsDiff" for m in model_names]].mean().mean()
    }

# === Build Interactive HTML ===
now = datetime.now().strftime("%Y-%m-%d %H:%M")

# Prepare JSON data for charts
model_names_json = json.dumps(model_names)
model_metrics_json = json.dumps({k: v for k, v in sorted_models})
commodity_options_json = json.dumps(unique_commodities)
market_options_json = json.dumps(unique_markets)
data_json = df.to_json(orient='records')

page_html = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>AgriCast360 — Price Predictions Report</title>
  <script src='https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js'></script>
  <script src='https://cdn.jsdelivr.net/npm/fuse.js@6.6.2/dist/fuse.min.js'></script>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{
      font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', 'Oxygen', sans-serif;
      background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
      padding: 20px;
      color: #333;
    }}
    .container {{
      max-width: 1600px;
      margin: 0 auto;
      background: white;
      border-radius: 12px;
      box-shadow: 0 10px 40px rgba(0, 0, 0, 0.1);
      padding: 30px;
    }}
    h1 {{
      font-size: 2.2em;
      color: #0d6efd;
      margin-bottom: 8px;
      font-weight: 700;
    }}
    .header-info {{
      color: #666;
      margin-bottom: 20px;
      font-size: 0.95em;
    }}
    .section {{
      margin: 30px 0;
      padding: 20px;
      background: #f8f9fa;
      border-radius: 8px;
      border-left: 4px solid #0d6efd;
    }}
    .section h2 {{
      font-size: 1.5em;
      color: #0d6efd;
      margin-bottom: 15px;
    }}
    .cards-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
      gap: 15px;
      margin-bottom: 20px;
    }}
    .card {{
      background: white;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 15px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.05);
      transition: all 0.3s ease;
    }}
    .card:hover {{
      transform: translateY(-2px);
      box-shadow: 0 4px 12px rgba(0,0,0,0.1);
    }}
    .card-rank {{
      display: inline-block;
      background: #0d6efd;
      color: white;
      padding: 2px 8px;
      border-radius: 12px;
      font-size: 0.8em;
      font-weight: bold;
      margin-right: 8px;
    }}
    .card-title {{
      font-weight: 600;
      margin: 8px 0;
      color: #333;
    }}
    .card-metric {{
      font-size: 1.8em;
      font-weight: bold;
      color: #0d6efd;
      margin: 5px 0;
    }}
    .card-label {{
      font-size: 0.85em;
      color: #777;
      text-transform: uppercase;
    }}
    .filters {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
      gap: 15px;
      margin-bottom: 20px;
    }}
    .filter-group {{
      display: flex;
      flex-direction: column;
    }}
    .filter-group label {{
      font-weight: 600;
      margin-bottom: 8px;
      color: #333;
      font-size: 0.9em;
    }}
    .filter-group select,
    .filter-group input {{
      padding: 10px 12px;
      border: 1px solid #ddd;
      border-radius: 6px;
      font-size: 0.95em;
      background: white;
    }}
    .filter-group select:focus,
    .filter-group input:focus {{
      outline: none;
      border-color: #0d6efd;
      box-shadow: 0 0 0 3px rgba(13, 110, 253, 0.1);
    }}
    .charts-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(500px, 1fr));
      gap: 20px;
      margin: 20px 0;
    }}
    .chart-container {{
      background: white;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 15px;
      position: relative;
      height: 400px;
    }}
    table {{
      width: 100%;
      border-collapse: collapse;
      margin-top: 15px;
      font-size: 0.95em;
    }}
    thead th {{
      background: #0d6efd;
      color: white;
      padding: 12px;
      text-align: left;
      font-weight: 600;
      position: sticky;
      top: 0;
      z-index: 10;
    }}
    tbody td {{
      padding: 10px 12px;
      border-bottom: 1px solid #eee;
    }}
    tbody tr:hover {{
      background: #f8f9fa;
    }}
    tbody tr:nth-child(even) {{
      background: #f8f9fa;
    }}
    .error-low {{ color: #28a745; font-weight: 600; }}
    .error-med {{ color: #ffc107; font-weight: 600; }}
    .error-high {{ color: #dc3545; font-weight: 600; }}
    .btn {{
      padding: 10px 20px;
      background: #0d6efd;
      color: white;
      border: none;
      border-radius: 6px;
      cursor: pointer;
      font-weight: 600;
      transition: all 0.2s;
    }}
    .btn:hover {{
      background: #0b5ed7;
      transform: translateY(-1px);
    }}
    .stats-row {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
      gap: 15px;
      margin-bottom: 20px;
    }}
    .stat-box {{
      background: white;
      padding: 15px;
      border-radius: 6px;
      border: 1px solid #ddd;
      text-align: center;
    }}
    .stat-value {{
      font-size: 1.6em;
      font-weight: bold;
      color: #0d6efd;
    }}
    .stat-label {{
      font-size: 0.85em;
      color: #666;
      margin-top: 5px;
      text-transform: uppercase;
    }}
  </style>
</head>
<body>
  <div class='container'>
    <h1>🌾 AgriCast360 — Price Predictions Report</h1>
    <div class='header-info'>
      <strong>Source:</strong> {os.path.basename(csv_path)} | 
      <strong>Records:</strong> {len(df):,} | 
      <strong>Generated:</strong> {now}
    </div>

    <!-- Model Rankings Section -->
    <div class='section'>
      <h2>📊 Model Performance Rankings (by MAE)</h2>
      <div class='cards-grid'>
"""

for rank, (model, metrics) in enumerate(sorted_models, 1):
    mae = metrics['MAE']
    rmse = metrics['RMSE']
    badge_color = "#28a745" if rank == 1 else "#0d6efd" if rank <= 3 else "#6c757d"
    page_html += f"""
        <div class='card'>
          <div style='display: flex; align-items: center; justify-content: space-between;'>
            <div style='background: {badge_color}; color: white; width: 32px; height: 32px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold;'>#{rank}</div>
            <div style='text-align: right;'>
              <div class='card-label'>Rank #{rank}</div>
            </div>
          </div>
          <div class='card-title'>{model}</div>
          <div class='card-metric'>{mae:,.1f}</div>
          <div class='card-label'>Mean Absolute Error (MAE)</div>
          <div class='card-metric' style='font-size: 1.2em; color: #6c757d;'>{rmse:,.1f}</div>
          <div class='card-label'>Root Mean Squared Error (RMSE)</div>
        </div>
"""

page_html += """
      </div>
    </div>

    <!-- Filters Section -->
    <div class='section'>
      <h2>🔍 Filters</h2>
      <div class='filters'>
        <div class='filter-group'>
          <label for='commodity-filter'>Commodity</label>
          <select id='commodity-filter'>
            <option value=''>All Commodities</option>
"""

for comm in unique_commodities:
    page_html += f"            <option value='{comm}'>{comm}</option>\n"

page_html += """
          </select>
        </div>
        <div class='filter-group'>
          <label for='market-filter'>Market</label>
          <select id='market-filter'>
            <option value=''>All Markets</option>
"""

for market in unique_markets:
    page_html += f"            <option value='{market}'>{market}</option>\n"

page_html += """
          </select>
        </div>
        <div class='filter-group'>
          <label for='search-filter'>Search (Commodity or Market)</label>
          <input type='text' id='search-filter' placeholder='Type to search...' />
        </div>
        <div class='filter-group'>
          <label>&nbsp;</label>
          <button class='btn' onclick='resetFilters()'>Reset Filters</button>
        </div>
      </div>
    </div>

    <!-- Summary Stats -->
    <div class='section'>
      <h2>📈 Summary Statistics</h2>
      <div class='stats-row'>
        <div class='stat-box'>
          <div class='stat-value'>{len(df):,}</div>
          <div class='stat-label'>Total Records</div>
        </div>
        <div class='stat-box'>
          <div class='stat-value'>{len(unique_commodities)}</div>
          <div class='stat-label'>Unique Commodities</div>
        </div>
        <div class='stat-box'>
          <div class='stat-value'>{len(unique_markets)}</div>
          <div class='stat-label'>Unique Markets</div>
        </div>
        <div class='stat-box'>
          <div class='stat-value'>{len(model_names)}</div>
          <div class='stat-label'>Models Evaluated</div>
        </div>
        <div class='stat-box'>
          <div class='stat-value'>₹{df['Actual_Price'].mean():,.0f}</div>
          <div class='stat-label'>Average Price</div>
        </div>
      </div>
    </div>

    <!-- Charts Section -->
    <div class='section'>
      <h2>📉 Performance Visualizations</h2>
      <div class='charts-grid'>
        <div class='chart-container'>
          <canvas id='mae-chart'></canvas>
        </div>
        <div class='chart-container'>
          <canvas id='rmse-chart'></canvas>
        </div>
        <div class='chart-container'>
          <canvas id='error-distribution-chart'></canvas>
        </div>
        <div class='chart-container'>
          <canvas id='commodity-price-chart'></canvas>
        </div>
      </div>
    </div>

    <!-- Data Table Section -->
    <div class='section'>
      <h2>📋 Detailed Results</h2>
      <div style='overflow-x: auto;'>
        <table id='results-table'>
          <thead>
            <tr>
              <th>Commodity</th>
              <th>Market</th>
              <th>Actual Price</th>
              <th>Best Prediction</th>
              <th>Best Error</th>
              <th>Worst Error</th>
            </tr>
          </thead>
          <tbody id='table-body'>
          </tbody>
        </table>
      </div>
    </div>

  </div>

  <script>
    // Data
    const allData = {data_json};
    const modelNames = {model_names_json};
    const modelMetrics = {model_metrics_json};
    const commodities = {commodity_options_json};
    const markets = {market_options_json};
    
    let filteredData = [...allData];
    let chartInstances = {{}};

    // Filter functions
    function applyFilters() {{
      const commodityFilter = document.getElementById('commodity-filter').value;
      const marketFilter = document.getElementById('market-filter').value;
      const searchFilter = document.getElementById('search-filter').value.toLowerCase();

      filteredData = allData.filter(row => {{
        const commodityMatch = !commodityFilter || row.Commodity === commodityFilter;
        const marketMatch = !marketFilter || row.Market === marketFilter;
        const searchMatch = !searchFilter || 
          row.Commodity.toLowerCase().includes(searchFilter) ||
          row.Market.toLowerCase().includes(searchFilter);
        return commodityMatch && marketMatch && searchMatch;
      }});

      updateTable();
      updateCharts();
    }}

    function resetFilters() {{
      document.getElementById('commodity-filter').value = '';
      document.getElementById('market-filter').value = '';
      document.getElementById('search-filter').value = '';
      filteredData = [...allData];
      updateTable();
      updateCharts();
    }}

    // Update table
    function updateTable() {{
      const tbody = document.getElementById('table-body');
      tbody.innerHTML = '';
      
      const displayed = filteredData.slice(0, 500); // Show first 500 for performance
      
      displayed.forEach(row => {{
        let bestError = Infinity;
        let worstError = -Infinity;
        
        modelNames.forEach(model => {{
          const absDiffKey = model + '_AbsDiff';
          const error = row[absDiffKey];
          bestError = Math.min(bestError, error);
          worstError = Math.max(worstError, error);
        }});

        const errorClass = bestError < 50 ? 'error-low' : bestError < 200 ? 'error-med' : 'error-high';
        
        const tr = document.createElement('tr');
        tr.innerHTML = `
          <td><strong>${{row.Commodity}}</strong></td>
          <td>${{row.Market}}</td>
          <td>₹${{row.Actual_Price.toLocaleString('en-IN')}}</td>
          <td>₹${{Math.round(row.Actual_Price - bestError).toLocaleString('en-IN')}}</td>
          <td><span class="${{errorClass}}">₹${{bestError.toFixed(1)}}</span></td>
          <td>₹${{worstError.toFixed(1)}}</td>
        `;
        tbody.appendChild(tr);
      }});
      
      if (displayed.length < filteredData.length) {{
        const tr = document.createElement('tr');
        tr.innerHTML = `<td colspan='6' style='text-align: center; color: #999;'>Showing ${{displayed.length}} of ${{filteredData.length}} records</td>`;
        tbody.appendChild(tr);
      }}
    }}

    // Update charts
    function updateCharts() {{
      updateMAEChart();
      updateRMSEChart();
      updateErrorDistribution();
      updateCommodityChart();
    }}

    function updateMAEChart() {{
      const ctx = document.getElementById('mae-chart').getContext('2d');
      const models = Object.keys(modelMetrics);
      const maes = models.map(m => modelMetrics[m].MAE);
      
      if (chartInstances.mae) chartInstances.mae.destroy();
      
      chartInstances.mae = new Chart(ctx, {{
        type: 'bar',
        data: {{
          labels: models.map(m => m.replace(/_/g, ' ')),
          datasets: [{{
            label: 'Mean Absolute Error',
            data: maes,
            backgroundColor: 'rgba(13, 110, 253, 0.7)',
            borderColor: 'rgba(13, 110, 253, 1)',
            borderWidth: 1
          }}]
        }},
        options: {{
          responsive: true,
          maintainAspectRatio: false,
          plugins: {{ legend: {{ display: false }} }},
          scales: {{ y: {{ beginAtZero: true }} }}
        }}
      }});
    }}

    function updateRMSEChart() {{
      const ctx = document.getElementById('rmse-chart').getContext('2d');
      const models = Object.keys(modelMetrics);
      const rmses = models.map(m => modelMetrics[m].RMSE);
      
      if (chartInstances.rmse) chartInstances.rmse.destroy();
      
      chartInstances.rmse = new Chart(ctx, {{
        type: 'line',
        data: {{
          labels: models.map(m => m.replace(/_/g, ' ')),
          datasets: [{{
            label: 'Root Mean Squared Error',
            data: rmses,
            borderColor: 'rgb(220, 53, 69)',
            backgroundColor: 'rgba(220, 53, 69, 0.1)',
            borderWidth: 2,
            fill: true,
            tension: 0.4
          }}]
        }},
        options: {{
          responsive: true,
          maintainAspectRatio: false,
          plugins: {{ legend: {{ display: true }} }},
          scales: {{ y: {{ beginAtZero: true }} }}
        }}
      }});
    }}

    function updateErrorDistribution() {{
      const ctx = document.getElementById('error-distribution-chart').getContext('2d');
      
      const errorRanges = [
        {{ min: 0, max: 50, label: '0-50' }},
        {{ min: 50, max: 100, label: '50-100' }},
        {{ min: 100, max: 200, label: '100-200' }},
        {{ min: 200, max: 500, label: '200-500' }},
        {{ min: 500, max: Infinity, label: '500+' }}
      ];
      
      const counts = errorRanges.map(range => {{
        return filteredData.filter(row => {{
          const bestError = Math.min(...modelNames.map(m => row[m + '_AbsDiff']));
          return bestError >= range.min && bestError < range.max;
        }}).length;
      }});
      
      if (chartInstances.errorDist) chartInstances.errorDist.destroy();
      
      chartInstances.errorDist = new Chart(ctx, {{
        type: 'doughnut',
        data: {{
          labels: errorRanges.map(r => r.label),
          datasets: [{{
            data: counts,
            backgroundColor: [
              'rgba(40, 167, 69, 0.7)',
              'rgba(255, 193, 7, 0.7)',
              'rgba(23, 162, 184, 0.7)',
              'rgba(253, 126, 20, 0.7)',
              'rgba(220, 53, 69, 0.7)'
            ]
          }}]
        }},
        options: {{
          responsive: true,
          maintainAspectRatio: false,
          plugins: {{ legend: {{ position: 'bottom' }} }}
        }}
      }});
    }}

    function updateCommodityChart() {{
      const ctx = document.getElementById('commodity-price-chart').getContext('2d');
      
      const commErrors = {{}};
      filteredData.forEach(row => {{
        if (!commErrors[row.Commodity]) commErrors[row.Commodity] = [];
        const bestError = Math.min(...modelNames.map(m => row[m + '_AbsDiff']));
        commErrors[row.Commodity].push(bestError);
      }});
      
      const commLabels = Object.keys(commErrors).slice(0, 15);
      const commAvgErrors = commLabels.map(c => 
        commErrors[c].reduce((a, b) => a + b, 0) / commErrors[c].length
      );
      
      if (chartInstances.commodity) chartInstances.commodity.destroy();
      
      chartInstances.commodity = new Chart(ctx, {{
        type: 'horizontalBar' in Chart.helpers ? 'bar' : 'bar',
        data: {{
          labels: commLabels,
          datasets: [{{
            label: 'Avg Error (Best Model)',
            data: commAvgErrors,
            backgroundColor: 'rgba(108, 117, 125, 0.7)',
            borderColor: 'rgba(108, 117, 125, 1)',
            borderWidth: 1
          }}]
        }},
        options: {{
          indexAxis: 'y',
          responsive: true,
          maintainAspectRatio: false,
          plugins: {{ legend: {{ display: false }} }},
          scales: {{ x: {{ beginAtZero: true }} }}
        }}
      }});
    }}

    // Event listeners
    document.getElementById('commodity-filter').addEventListener('change', applyFilters);
    document.getElementById('market-filter').addEventListener('change', applyFilters);
    document.getElementById('search-filter').addEventListener('input', applyFilters);

    // Initialize
    updateTable();
    updateCharts();
  </script>
</body>
</html>
"""

# Save HTML
with open(html_path, "w", encoding="utf-8") as f:
    f.write(page_html)

print(f"✅ Enhanced report generated: {html_path}")
print(f"📊 Features: Model rankings, interactive filters, 4 performance charts, detailed table")

# Show a clickable link inside the notebook (if supported)
from IPython.display import HTML
html_path_url = html_path.replace('\\', '/')
HTML(f"<a href='file:///{html_path_url}' target='_blank' style='font-size:16px; padding:10px 20px; background:#0d6efd; color:white; text-decoration:none; border-radius:5px;'>🚀 Open Enhanced Report</a>")

✅ Loaded 16,910 records | 30 columns | 90 commodities | 34 markets
✅ COMPREHENSIVE report generated with ALL information!
📍 Saved to: D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Model_Results\price_predictions_report.html
📊 Includes: Model rankings, commodity analysis, best/worst predictions, detailed explorer, 4 charts
✅ COMPREHENSIVE report generated with ALL information!
📍 Saved to: D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Model_Results\price_predictions_report.html
📊 Includes: Model rankings, commodity analysis, best/worst predictions, detailed explorer, 4 charts


✅ Loaded 16,910 records | 30 columns | 90 commodities | 34 markets
✅ COMPREHENSIVE report generated with ALL information!
📍 Saved to: D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Model_Results\price_predictions_report.html
📊 Includes: Model rankings, commodity analysis, best/worst predictions, detailed explorer, 4 charts
✅ COMPREHENSIVE report generated with ALL information!
📍 Saved to: D:\CUDA_Experiments\Git_HUB\Machine-Learning\AgriCast360\AgriCast360_V2\Model_Results\price_predictions_report.html
📊 Includes: Model rankings, commodity analysis, best/worst predictions, detailed explorer, 4 charts
